# Gold - fact_component_production

This notebook loads `amperedrive_giga.gold.fact_component_production`, one row per component production run (`component_run_id`). Sources are `silver.fact_component_production`, filtered to active SCD2 versions (`is_current = TRUE`), joined to `silver.dim_products` for cost and cycle-time metrics.

* Load pattern: Incremental MERGE on the business key, with a watermark on `silver.updated_at` compared with `MAX(gold.gold_load_ts)`.
* DQ-G02: duplicate active runs stop the MERGE (SCD2 leak check).
* DQ-G03: rows where `quantity_rejected > quantity_produced` are reported as warnings.
* Foreign surrogate keys use -1 (unknown member) when the natural key is NULL, because `xxhash64(NULL)` returns 42 rather than NULL.
* Load after the gold dimension notebooks.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.fact_component_production (
  production_sk            BIGINT         NOT NULL COMMENT 'PK - xxhash64(component_run_id)',
  component_run_id         INT            NOT NULL COMMENT 'NK / degenerate dimension',
  production_date_key      INT            NOT NULL COMMENT 'FK -> dim_date.date_key (start_time)',
  plant_sk                 BIGINT         NOT NULL COMMENT 'FK -> dim_plant',
  product_sk               BIGINT         NOT NULL COMMENT 'FK -> dim_product',
  equipment_sk             BIGINT         NOT NULL COMMENT 'FK -> dim_equipment',
  quantity_produced        INT            NOT NULL,
  quantity_rejected        INT            NOT NULL,
  good_quantity            INT            NOT NULL,
  downtime_minutes         INT            NOT NULL,
  first_pass_yield_pct     DECIMAL(5,2)   NOT NULL,
  scrap_cost_usd           DECIMAL(38,18) NOT NULL,
  total_output_value_usd   DECIMAL(38,18) NOT NULL,
  actual_duration_secs     INT,
  cycle_time_variance_secs INT,
  gold_load_ts             TIMESTAMP      NOT NULL,
  CONSTRAINT pk_fact_component_production PRIMARY KEY (production_sk)
)
COMMENT 'Gold component production fact (active SCD2 versions from silver.fact_component_production)'

In [0]:
%sql
-- Watermark = last gold load time; silver rows stamped (updated_at) after it are reprocessed
DECLARE OR REPLACE VARIABLE v_watermark TIMESTAMP;
SET VAR v_watermark = (
  SELECT COALESCE(MAX(gold_load_ts), TIMESTAMP'1900-01-01 00:00:00')
  FROM amperedrive_giga.gold.fact_component_production
);
SELECT v_watermark AS watermark;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_fact_component_production AS
WITH active AS (
  -- SCD Type 2 filtering rule: active records only
  SELECT *
  FROM amperedrive_giga.silver.fact_component_production
  WHERE is_current = TRUE
    AND updated_at > v_watermark
),
base AS (
  SELECT
    a.*,
    COALESCE(a.quantity_produced, 0)                                      AS qty_produced,
    COALESCE(a.quantity_rejected, 0)                                      AS qty_rejected,
    GREATEST(COALESCE(a.quantity_produced, 0) - COALESCE(a.quantity_rejected, 0), 0) AS good_qty,
    CAST(UNIX_TIMESTAMP(a.end_time) - UNIX_TIMESTAMP(a.start_time) AS INT) AS actual_dur_secs
  FROM active a
)
SELECT
  xxhash64(CAST(b.component_run_id AS STRING))                                         AS production_sk,
  b.component_run_id,
  CAST(DATE_FORMAT(b.start_time, 'yyyyMMdd') AS INT)                                   AS production_date_key,
  CASE WHEN b.plant_id     IS NULL THEN -1 ELSE xxhash64(CAST(b.plant_id     AS STRING)) END AS plant_sk,
  CASE WHEN b.product_id   IS NULL THEN -1 ELSE xxhash64(CAST(b.product_id   AS STRING)) END AS product_sk,
  CASE WHEN b.equipment_id IS NULL THEN -1 ELSE xxhash64(CAST(b.equipment_id AS STRING)) END AS equipment_sk,
  b.qty_produced                                                                       AS quantity_produced,
  b.qty_rejected                                                                       AS quantity_rejected,
  b.good_qty                                                                           AS good_quantity,
  COALESCE(b.downtime_minutes, 0)                                                      AS downtime_minutes,
  CAST(COALESCE(ROUND((b.good_qty / NULLIF(b.qty_produced, 0)) * 100, 2), 0) AS DECIMAL(5,2)) AS first_pass_yield_pct,
  CAST(COALESCE(ROUND(b.qty_rejected * p.material_cost_usd, 2), 0) AS DECIMAL(38,18))  AS scrap_cost_usd,
  CAST(COALESCE(ROUND(b.good_qty * p.factory_output_cost_usd, 2), 0) AS DECIMAL(38,18)) AS total_output_value_usd,
  b.actual_dur_secs                                                                    AS actual_duration_secs,
  b.actual_dur_secs - p.standard_cycle_time_secs                                       AS cycle_time_variance_secs,
  CURRENT_TIMESTAMP()                                                                  AS gold_load_ts
FROM base b
LEFT JOIN amperedrive_giga.silver.dim_products p
  ON b.product_id = p.product_id;

SELECT COUNT(*) AS incoming_rows FROM vw_src_fact_component_production;

In [0]:
%sql
-- DQ-G02: COUNT(component_run_id) = COUNT(DISTINCT component_run_id) -> Fail MERGE (SCD2 leak check)
SELECT
  COUNT(component_run_id)          AS total_runs,
  COUNT(DISTINCT component_run_id) AS distinct_runs,
  CASE WHEN COUNT(component_run_id) <> COUNT(DISTINCT component_run_id)
       THEN raise_error('DQ-G02 failed: duplicate active component_run_id in silver (SCD2 leak)')
  END AS dq_g02_status
FROM vw_src_fact_component_production

In [0]:
%sql
-- DQ-G03: quantity_rejected <= quantity_produced -> Log Warning & Flag Record (load continues)
SELECT
  component_run_id,
  quantity_produced,
  quantity_rejected,
  'DQ-G03 WARNING: quantity_rejected > quantity_produced' AS dq_flag
FROM vw_src_fact_component_production
WHERE quantity_rejected > quantity_produced

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.fact_component_production AS t
USING vw_src_fact_component_production AS s
ON t.component_run_id = s.component_run_id
WHEN MATCHED AND (
     NOT (t.production_date_key      <=> s.production_date_key)
  OR NOT (t.plant_sk                 <=> s.plant_sk)
  OR NOT (t.product_sk               <=> s.product_sk)
  OR NOT (t.equipment_sk             <=> s.equipment_sk)
  OR NOT (t.quantity_produced        <=> s.quantity_produced)
  OR NOT (t.quantity_rejected        <=> s.quantity_rejected)
  OR NOT (t.downtime_minutes         <=> s.downtime_minutes)
  OR NOT (t.scrap_cost_usd           <=> s.scrap_cost_usd)
  OR NOT (t.total_output_value_usd   <=> s.total_output_value_usd)
  OR NOT (t.actual_duration_secs     <=> s.actual_duration_secs)
  OR NOT (t.cycle_time_variance_secs <=> s.cycle_time_variance_secs)
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
from delta.tables import DeltaTable

source_table = "amperedrive_giga.silver.fact_component_production"
staged_view = "vw_src_fact_component_production"
target_fqn = "amperedrive_giga.gold.fact_component_production"

rows_extracted = spark.table(source_table).count()  # rows read from silver source (all SCD2 versions)
rows_loaded = spark.table(staged_view).count()      # active, incremental rows staged for the MERGE

# MERGE metrics: _sqldf holds the result of the MERGE cell above; fall back to the latest MERGE in Delta history
try:
    m = _sqldf.first().asDict()
    rows_inserted, rows_updated = int(m["num_inserted_rows"]), int(m["num_updated_rows"])
except Exception:
    h = (DeltaTable.forName(spark, target_fqn).history(10)
         .filter("operation = 'MERGE'").orderBy("version", ascending=False).first())
    om = h["operationMetrics"] if h else {}
    rows_inserted = int(om.get("numTargetRowsInserted", 0))
    rows_updated = int(om.get("numTargetRowsUpdated", 0))

result = {
    "rows_extracted": str(rows_extracted),
    "rows_loaded": str(rows_loaded),
    "rows_inserted": str(rows_inserted),
    "rows_updated": str(rows_updated)
}

dbutils.notebook.exit(result)